In [1]:
from qiskit.circuit import Parameter, QuantumCircuit, QuantumRegister, ClassicalRegister

from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector, Operator
from scipy.optimize import minimize 
from qiskit.circuit.library import QFT
from qiskit import transpile
from qiskit.circuit.library import UnitaryGate

import random
import matplotlib.pyplot as plt
import scipy.linalg as scl
import numpy as np
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeKyiv
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()
# backend=FakeKyiv()
# sampler = Sampler(backend = backend)
pm = generate_preset_pass_manager(backend=backend,optimization_level=2)

In [2]:


nb_qubits = 3

N = 2**nb_qubits
m = np.zeros((N,N))
for j in range(N):
    if j == N-1:
        break
    else:
       m[j,j+1] = -1 

for j in range(N):
    if j == N-1:
        break
    else:
       m[j+1,j] = -1 
for j in range(N):
   m[j,j] = 2 
m[0] = np.array([1]+ [0]*(N-1))
m[1,0] = 0

b = np.array([0,0.25,0.25,0.25,0.25,0.5,0.5,0.5])
m

array([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  2., -1.,  0.,  0.,  0.,  0.,  0.],
       [ 0., -1.,  2., -1.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -1.,  2., -1.,  0.,  0.,  0.],
       [ 0.,  0.,  0., -1.,  2., -1.,  0.,  0.],
       [ 0.,  0.,  0.,  0., -1.,  2., -1.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -1.,  2., -1.],
       [ 0.,  0.,  0.,  0.,  0.,  0., -1.,  2.]])

In [3]:
d,u = np.linalg.eig(m)
k = np.max(d)/np.min(d)
dm = max(d)
# m = m/dm

In [4]:
np.linalg.det(m)



np.float64(7.999999999999998)

In [5]:
def U_b(nb_qubits):
    circ = QuantumCircuit(nb_qubits)
    circ.prepare_state(b)
    return circ
U = U_b(nb_qubits)
b = np.array(Statevector(U_b(nb_qubits)))
b

array([1.66533454e-16+5.55111512e-17j, 2.50000000e-01+5.06539255e-16j,
       2.50000000e-01-2.77555756e-16j, 2.50000000e-01-3.33066907e-16j,
       2.50000000e-01-5.55111512e-16j, 5.00000000e-01-6.66133815e-16j,
       5.00000000e-01+7.21644966e-16j, 5.00000000e-01+7.77156117e-16j])

In [6]:
def Hamiltonian(m):
    Ub = np.array(Operator(U_b(nb_qubits)))
    z = np.array([[1,0],
                 [0,-1]])
    I = np.array([[1,0],
                 [0,1]])
    def tensor(j,k,l):
        return np.kron(j,np.kron(k,l))
    M1 = (np.dot(np.dot(Ub,tensor(z,I,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,z,I)),np.conj(Ub.T))
          +np.dot(np.dot(Ub,tensor(I,I,z)),np.conj(Ub.T)))
    M = 0.5*np.dot(np.dot(np.conj(m.T),(tensor(I,I,I) - M1/nb_qubits)),m)

    return M
A = Hamiltonian(m)

In [7]:
U1=scl.expm(2**0*2*np.pi*1j*A) 
U2=scl.expm(2**1*2*np.pi*1j*A) 
U3=scl.expm(2**2*2*np.pi*1j*A) 
U4=scl.expm(2**3*2*np.pi*1j*A) 
U5=scl.expm(2**4*2*np.pi*1j*A) 
U6=scl.expm(2**5*2*np.pi*1j*A) 
U7=scl.expm(2**6*2*np.pi*1j*A) 
U8=scl.expm(2**7*2*np.pi*1j*A)
 
u1gate = UnitaryGate(U1)
u2gate = UnitaryGate(U2)
u3gate=UnitaryGate(U3)
u4gate=UnitaryGate(U4)
u5gate=UnitaryGate(U5)
u6gate=UnitaryGate(U6)
u7gate=UnitaryGate(U7)
u8gate=UnitaryGate(U8)


C_u1gate=u1gate.control()
C_u2gate=u2gate.control()
C_u3gate=u3gate.control()
C_u4gate=u4gate.control()
C_u5gate=u5gate.control()
C_u6gate=u6gate.control()
C_u7gate=u7gate.control()
C_u8gate=u8gate.control()

In [8]:
x_exact = np.linalg.solve(m,b)
x_exact = x_exact/np.linalg.norm(x_exact)
nb_qubits = 3
depth = 2
qubits = list(range(nb_qubits))
N = len(qubits)
nb_params = int(9*N*depth)
RMSE = []
rep = 1000
for _ in range(rep):

    Parameters = np.array([random.random() for _ in range(0, nb_params)])
    def ansatz(Parameters):
        qc = QuantumCircuit(N)
        for d in range(depth):
            param1=Parameters[d*9*N:(d+1)*(9*N)]
            for q in range(N):
                qc.ry(param1[q],qubits[q])
                qc.ry(param1[q+N],qubits[q])
                qc.ry(param1[q+2*N],qubits[q])
            qc.barrier()
            for q in range(N):
                qc.cx(qubits[q], qubits[(q+1)% N])
                qc.ry(param1[q+3*N],qubits[q])
                qc.ry(param1[q+4*N],qubits[(q+1)% N])
                qc.cx(qubits[(q+1)% N], qubits[q])
                qc.ry(param1[q+5*N],qubits[(q+1)% N])
                qc.cx(qubits[q], qubits[(q+1)% N])
            qc.barrier()
            if d==depth-1:
                for q in range(N):
                    qc.ry(param1[q+6*N],qubits[q])
                    qc.ry(param1[q+7*N],qubits[q])
                    qc.ry(param1[q+8*N],qubits[q])
            qc.barrier()
        
        return qc    
    
    
    def circ(parameters):
        x=QuantumRegister(11)
        c=ClassicalRegister(8)
        circuit = QuantumCircuit(x,c)
        phi=parameters
        circuit=circuit.compose(ansatz(parameters),x[8:11])
        circuit.h(x[0:8]) 
        circuit.append(C_u1gate, [x[0],x[8],x[9],x[10]])    
        circuit.append(C_u2gate, [x[1],x[8],x[9],x[10]])    
        circuit.append(C_u3gate, [x[2],x[8],x[9],x[10]])    
        circuit.append(C_u4gate, [x[3],x[8],x[9],x[10]]) 
        circuit.append(C_u5gate, [x[4],x[8],x[9],x[10]])
        circuit.append(C_u6gate, [x[5],x[8],x[9],x[10]])    
        circuit.append(C_u7gate, [x[6],x[8],x[9],x[10]]) 
        circuit.append(C_u8gate, [x[7],x[8],x[9],x[10]])
        circuit &= QFT(num_qubits = 8, approximation_degree = 0, do_swaps = True, 
                       inverse = True, insert_barriers = False, name='qft')
        circuit.measure(x[0:8],c)       
        return circuit
    
    shots = 100000
    def cost(Parameters):

        job = backend.run(pm.run(circ(Parameters)),shots = shots).result()
        result = job.get_counts(0)
        if '00000000'not in result: 
            res = 1
        else:
            res = 1 - result['00000000']/shots
        return res
    # cost(parameters)

    def Optimizer(fun, x0, args=(), maxfev=None, 
                  reset_interval=None, eps=None, callback=None, **_):
        
        x0 = np.asarray(x0)
        recycle_z0 = None
        niter = 0
        funcalls = 0
    
        while True:
    
            idx = niter % x0.size
    
            if reset_interval > 0:
                if niter % reset_interval == 0:
                    recycle_z0 = None
    
            if recycle_z0 is None:
                z0 = fun(np.copy(x0), *args)
                funcalls += 1
            else:
                z0 = recycle_z0
    
            p = np.copy(x0)
            p[idx] = x0[idx] + np.pi / 2
            z1 = fun(p, *args)
            funcalls += 1
    
            p = np.copy(x0)
            p[idx] = x0[idx] - np.pi / 2
            z3 = fun(p, *args)
            funcalls += 1
    
            z2 = z1 + z3 - z0
            c = (z1 + z3) / 2
            a = np.sqrt((z0 - z2) ** 2 + (z1 - z3) ** 2) / 2
            b = np.arctan((z1 - z3) / ((z0 - z2) + 1e-32 * (z0 == z2))) + x0[idx]
            b += 0.5 * np.pi + 0.5 * np.pi * np.sign((z0 - z2) + eps * (z0 == z2))
            x0[idx] = b
            recycle_z0 = c - a
            if callback is not None:
                callback(np.copy(x0))
            if funcalls >= maxfev:
                break
            niter += 1
        # return OptimizeResult(fun=problabel0(np.copy(x0)), x=x0, nit=niter, 
        #                       nfev=funcalls, success=(niter > 1))
    
    def save(Parameters):
        global Cost,Params
        Cost.append(cost(Parameters))
        Params.append(Parameters)
        # print(cost(Parameters))
    Cost = []
    Params = []
    Optimizer(cost, Parameters, args=(), maxfev = 4000, 
              reset_interval = 32, eps=1e-32, callback=save)


    e = []
    F = []
    norm_e = []
    for k in range(len(Params)):
        state = np.array(Statevector(ansatz(Params[k])))
        norm = np.dot(state,x_exact)
        e.append(x_exact - state/norm)
        f = abs(np.dot(state,x_exact))**2
        F.append(f)

    for v in e:
        norm_e.append(float(np.linalg.norm(v)))
    Res = norm_e[np.argmax(F)]
    RMSE.append(Res)
print(RMSE)
float(np.mean(RMSE))

[0.0007493452958798124, 0.0008152572519855624, 0.0007506272862708628, 0.0008043753189914018, 0.0007637766728566494, 0.0007918254897183166, 0.0007909108493078568, 0.0007388103844220837, 0.0007637812003196599, 0.0007177929779515168, 0.0008063917075329968, 0.0007469867703918599, 0.0008051335423001384, 0.0007328735236479228, 0.0007443724838634941, 0.0007647312476467017, 0.0008016789760443708, 0.0007899653563857059, 0.0007218631581949835, 0.0007427061420885061, 0.0007847749088439627, 0.0007292863352927918, 0.0007790190152559626, 0.0007939243105098223, 0.0007702276414412588, 0.0007637046136197462, 0.0008045376740820694, 0.0007561847745246844, 0.0007507232710425397, 0.0007300675435218186, 0.0007605775529847577, 0.0007621979035562098, 0.0007272287876942316, 0.000747184988212403, 0.0007316996515552584, 0.0007248498903630064, 0.000733087710510126, 0.0007266803226774882, 0.0008123609076168447, 0.0007206812621051644, 0.0007999173489613501, 0.0007832240695690101, 0.0007936591774306589, 0.0007841774

0.0007671415014498497

In [10]:

float(np.mean(RMSE))

0.0007671415014498497